<a href="https://colab.research.google.com/github/shabir-mp/Collab-Project/blob/v2/trial2_penyisihanAIrena.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
# ============================================
# XGBOOST LEVEL 2 - AIrena Competition
# ============================================

import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import re

# Import the EarlyStopping callback (removed as callbacks are not supported directly in fit for this XGBoost version)

# ============================
# LOAD DATA
# ============================
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

TARGET = "penghargaan_misi"

# ============================
# DETECT TIME COLUMNS
# format contoh: "54d 13:55:29.128"
# ============================
def convert_time(s):
    if isinstance(s, float) or pd.isna(s):
        return np.nan
    m = re.match(r"(\d+)d\s+(\d+):(\d+):(\d+)\.(\d+)", s)
    if m:
        d, h, mi, sec, ms = map(int, m.groups())
        return d*86400 + h*3600 + mi*60 + sec + ms/1000
    return np.nan

time_cols_from_train = [col for col in train.columns if "waktu" in col.lower() or "durasi" in col.lower()]

for c in time_cols_from_train:
    if c in train.columns:
        train[c] = train[c].apply(convert_time)
    if c in test.columns:
        test[c] = test[c].apply(convert_time)

# ============================
# Label Encode Target
# ============================
le_target = LabelEncoder()
train[TARGET] = le_target.fit_transform(train[TARGET])

# ============================
# Encode All Categorical Columns
# ============================
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
cat_cols = [c for c in cat_cols if c != TARGET]

encoders = {}
for c in cat_cols:
    enc = LabelEncoder()
    # Fit the encoder on combined unique values from both train and test
    combined_data = pd.concat([train[c].astype(str), test[c].astype(str)], axis=0).unique()
    enc.fit(combined_data)

    train[c] = enc.transform(train[c].astype(str))
    if c in test.columns: # Ensure column exists in test before transforming
        test[c] = enc.transform(test[c].astype(str))
    encoders[c] = enc

# ============================
# Align features between train and test
# ============================
# Identify features in training set (excluding target)
train_features = [col for col in train.columns if col != TARGET]

# Identify common features present in both train and test sets
common_features = list(set(train_features) & set(test.columns))

# Filter X and X_test to only include common features
X = train[common_features]
y = train[TARGET]
X_test = test[common_features]

# ============================
# K-FOLD TRAINING
# ============================
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

test_pred_proba = np.zeros((len(test), len(le_target.classes_)))
oof = np.zeros(len(train))

# MODEL PARAMETER (OPTIMIZED)
xgb_params = {
    "objective": "multi:softprob",
    "num_class": len(le_target.classes_),
    "eval_metric": "mlogloss",
    "max_depth": 8,
    "learning_rate": 0.05,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "n_estimators": 1500,
    "tree_method": "hist"
}

fold_idx = 1

for train_idx, val_idx in kf.split(X, y):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBClassifier(**xgb_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)]
        # Removed early_stopping_rounds and verbose as they are not accepted by this XGBoost version
    )

    # predict oof
    val_proba = model.predict_proba(X_val)
    val_pred = np.argmax(val_proba, axis=1)
    oof[val_idx] = val_pred

    f1 = f1_score(y_val, val_pred, average="macro")
    print(f"Fold {fold_idx} F1-macro: {f1:.4f}")
    fold_idx += 1

    # accumulate test predictions
    test_pred_proba += model.predict_proba(X_test)

# average from folds
test_pred_proba /= kf.n_splits
test_pred = np.argmax(test_pred_proba, axis=1)

# ============================
# OOF F1
# ============================
oof_f1 = f1_score(y, oof, average="macro")
print("\nOOF F1-macro:", round(oof_f1, 4))

# ============================
# CREATE SUBMISSION
# ============================
test_pred_labels = le_target.inverse_transform(test_pred)

submission = pd.DataFrame({
    "id": test["id"],
    "penghargaan_misi": test_pred_labels
})

submission.to_csv("submission_xgb_level2.csv", index=False)
print("Saved as submission_xgb_level2.csv")

submission.head()

Streaming output truncated to the last 5000 lines.
[1007]	validation_0-mlogloss:0.85456
[1008]	validation_0-mlogloss:0.85484
[1009]	validation_0-mlogloss:0.85489
[1010]	validation_0-mlogloss:0.85502
[1011]	validation_0-mlogloss:0.85524
[1012]	validation_0-mlogloss:0.85531
[1013]	validation_0-mlogloss:0.85526
[1014]	validation_0-mlogloss:0.85553
[1015]	validation_0-mlogloss:0.85567
[1016]	validation_0-mlogloss:0.85567
[1017]	validation_0-mlogloss:0.85597
[1018]	validation_0-mlogloss:0.85618
[1019]	validation_0-mlogloss:0.85643
[1020]	validation_0-mlogloss:0.85647
[1021]	validation_0-mlogloss:0.85654
[1022]	validation_0-mlogloss:0.85673
[1023]	validation_0-mlogloss:0.85696
[1024]	validation_0-mlogloss:0.85702
[1025]	validation_0-mlogloss:0.85722
[1026]	validation_0-mlogloss:0.85737
[1027]	validation_0-mlogloss:0.85759
[1028]	validation_0-mlogloss:0.85776
[1029]	validation_0-mlogloss:0.85799
[1030]	validation_0-mlogloss:0.85817
[1031]	validation_0-mlogloss:0.85832
[1032]	validation_0-mlog

,id,penghargaan_misi
0,1,Mission Incomplete
1,2,Silver
2,3,Bronze
3,4,Bronze
4,5,Bronze
